In [3]:
import pandas as pd
df = pd.read_csv('../data/raw/loan.csv', low_memory=False, nrows=100000)
df.shape
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Columns: 145 entries, id to settlement_term
dtypes: float64(58), int64(52), str(35)
memory usage: 126.1 MB


In [4]:
df['loan_status'].value_counts()


loan_status
Current               96793
Fully Paid             2431
Late (31-120 days)      318
In Grace Period         316
Late (16-30 days)       123
Charged Off              19
Name: count, dtype: int64

In [6]:
import pandas as pd

useful_cols = ['loan_amnt', 'term', 'int_rate', 'installment', 'grade', 'sub_grade',
               'emp_length', 'home_ownership', 'annual_inc', 'purpose', 'dti',
               'open_acc', 'revol_bal', 'revol_util', 'total_acc', 'loan_status']

chunks = []
for chunk in pd.read_csv('../data/raw/loan.csv', low_memory=False, usecols=useful_cols, chunksize=200000):
    chunks.append(chunk)
    print("Loaded a chunk, total so far:", sum(len(c) for c in chunks))

df_full = pd.concat(chunks, ignore_index=True)
df_full.shape

Loaded a chunk, total so far: 200000
Loaded a chunk, total so far: 400000
Loaded a chunk, total so far: 600000
Loaded a chunk, total so far: 800000
Loaded a chunk, total so far: 1000000
Loaded a chunk, total so far: 1200000
Loaded a chunk, total so far: 1400000
Loaded a chunk, total so far: 1600000
Loaded a chunk, total so far: 1800000
Loaded a chunk, total so far: 2000000
Loaded a chunk, total so far: 2200000
Loaded a chunk, total so far: 2260668


(2260668, 16)

In [7]:
df_full['loan_status'].value_counts()

loan_status
Fully Paid                                             1041952
Current                                                 919695
Charged Off                                             261655
Late (31-120 days)                                       21897
In Grace Period                                           8952
Late (16-30 days)                                         3737
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     31
Name: count, dtype: int64

In [8]:
df_model = df_full[df_full['loan_status'].isin(['Fully Paid', 'Charged Off'])].copy()
df_model['default'] = (df_model['loan_status'] == 'Charged Off').astype(int)

print(df_model.shape)
print(df_model['default'].value_counts())

(1303607, 17)
default
0    1041952
1     261655
Name: count, dtype: int64


In [9]:
df_model.to_csv('../data/processed/loan_cleaned.csv', index=False)
print("Saved successfully!")

Saved successfully!


## Target Variable Definition
We kept only loans with a finished outcome: "Fully Paid" (0) and "Charged Off" (1).
Excluded: "Current", "Late", "In Grace Period", "Default" (rare), and "Does not meet credit policy" rows,
since these don't have a confirmed final outcome yet or are too ambiguous.
Final dataset: 1,303,607 rows. Class distribution: 80% paid, 20% defaulted (imbalanced).
